# Phase 04 — LoRA fine-tune: financial sentiment classifier

Paper Trading Agent project. Replaces the `sentiment_analyst` node's GPT-4o-mini
zero-shot API call with a small, locally-run, LoRA-fine-tuned LLM — the
"LLM fine-tuning" resume gap the project set out to close.

**Why Financial PhraseBank instead of our own backtest history:** the
project's own collected sentiment calls (from backtests 3, 4, 6 in the
paper-trading-agent repo) are ~96% HOLD — real financial headlines mostly
aren't dramatic, and the walk-forward backtests only ran on one symbol
(AAPL) over ~19 months. That's nowhere near enough BUY/SELL examples to
teach a model the distinction. Financial PhraseBank
(https://huggingface.co/datasets/takala/financial_phrasebank) is a public,
human-annotated financial sentiment dataset (3,453 sentences at 75%+
annotator agreement) with a much healthier class balance. We train on
that, then sanity-check transfer to our own real headlines at the end.

**Model:** Qwen2.5-1.5B-Instruct, QLoRA (4-bit base + LoRA adapters) — fits
comfortably on a Kaggle T4 (16GB).

**Output contract:** the model is trained to emit the same JSON shape
`sentiment_analyst.py`'s `SentimentCall` already expects —
`{"opinion": "BUY"|"HOLD"|"SELL", "confidence": float, "reasoning": str}`
— so the adapter can be swapped in as a drop-in local alternative to the
OpenAI call, no changes needed to the node's calling code, only to which
backend it points at.

**Runtime:** Kaggle notebook settings → Accelerator → GPU T4 x2 (or P100).
Internet must be ON (Settings → Internet) to pull the base model and
dataset from Hugging Face.


In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

## 1. Load and prep Financial PhraseBank

Loading from `lmassaron/FinancialPhraseBank` — a plain Parquet re-upload
of the same Financial PhraseBank data (Malo et al. 2014), with train/
validation/test splits already made (3,870 / 484 / 484). The canonical
`takala/financial_phrasebank` repo still ships an old Python loading
script, and current `datasets` versions have dropped script support
entirely (`RuntimeError: Dataset scripts are no longer supported`) — this
mirror sidesteps that with no loading-script dependency.

Label mapping to this project's action space: `positive` → BUY,
`negative` → SELL, `neutral` → HOLD. Same three-way mapping
`sentiment_analyst.py`'s prompt already asks GPT-4o-mini to make, just
grounded here in human-annotated data instead of a zero-shot call.

In [ ]:
from datasets import load_dataset

ds = load_dataset("lmassaron/FinancialPhraseBank")
print(ds)

LABEL_MAP = {0: "SELL", 1: "HOLD", 2: "BUY"}  # negative, neutral, positive
from collections import Counter
print("train:", Counter(LABEL_MAP[x] for x in ds["train"]["label"]))
print("validation:", Counter(LABEL_MAP[x] for x in ds["validation"]["label"]))

In [ ]:
SYSTEM_PROMPT = (
    "You are a sentiment analyst for a stock paper-trading system. You will "
    "be given a list of recent news headlines for one symbol. Judge whether "
    "the overall tone of the headlines is bullish, bearish, or neutral for "
    "the stock's near-term price, and how confident you are in that read.\n\n"
    "Rules:\n"
    "- Base your judgment ONLY on the headlines given. Do not use outside "
    "knowledge of the company or assume information not present in the text.\n"
    "- If headlines are mixed, contradictory, or mostly routine/non-market-moving, "
    "prefer HOLD with lower confidence over guessing a direction.\n"
    "- confidence must be a number between 0.0 and 1.0.\n"
    "- reasoning must be 1-3 sentences, in plain English, that a human could "
    "audit against the headlines shown."
)

# PhraseBank has no confidence/reasoning fields (it's a label-only dataset).
# confidence is fixed at a plausible constant and reasoning is a short
# templated justification -- what's actually being supervised, and the
# thing that matters for integration, is the `opinion` field. Confidence/
# reasoning quality is inherited from Qwen2.5-Instruct's own instruction-
# following, not fit to PhraseBank targets.
REASONING_TEMPLATES = {
    "BUY": "The headline reads as positive for the company's near-term outlook.",
    "SELL": "The headline reads as negative for the company's near-term outlook.",
    "HOLD": "The headline is largely neutral or routine, without a clear directional signal.",
}

def to_example(sentence, label_id):
    opinion = LABEL_MAP[label_id]
    user = f"Symbol: (unspecified)\nHeadlines (1):\n- {sentence.strip()}"
    target = {
        "opinion": opinion,
        "confidence": 0.75,
        "reasoning": REASONING_TEMPLATES[opinion],
    }
    return {"system": SYSTEM_PROMPT, "user": user, "target": json.dumps(target)}

import json
train_examples = [to_example(s, l) for s, l in zip(ds["train"]["sentence"], ds["train"]["label"])]
val_examples = [to_example(s, l) for s, l in zip(ds["validation"]["sentence"], ds["validation"]["label"])]
print(f"train: {len(train_examples)}  val: {len(val_examples)}")
print(train_examples[0])

## 2. Load Qwen2.5-1.5B-Instruct in 4-bit + attach LoRA adapters

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Format as chat examples and fine-tune with `trl.SFTTrainer`

Each example is formatted with Qwen's chat template (system + user +
assistant turns) so the model sees exactly the input shape
`sentiment_analyst.py` sends at inference time, with the assistant turn
being the JSON target it should learn to reproduce.

In [ ]:
from datasets import Dataset

def format_chat(ex):
    messages = [
        {"role": "system", "content": ex["system"]},
        {"role": "user", "content": ex["user"]},
        {"role": "assistant", "content": ex["target"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds = Dataset.from_list(train_examples).map(format_chat, remove_columns=["system","user","target"])
val_ds = Dataset.from_list(val_examples).map(format_chat, remove_columns=["system","user","target"])
print(train_ds[0]["text"][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="/kaggle/working/qwen-sentiment-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=25,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)
trainer.train()

In [ ]:
ADAPTER_DIR = "/kaggle/working/qwen-sentiment-lora-final"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import shutil
shutil.make_archive("/kaggle/working/qwen-sentiment-lora-adapter", "zip", ADAPTER_DIR)
print("Zipped adapter ready in the Output tab: qwen-sentiment-lora-adapter.zip")
print("Download it from Kaggle's Output panel after the run finishes.")

## 4. Evaluate: held-out PhraseBank accuracy (the `opinion` field only)

This is the metric that actually matters for integration — does the
model reliably land on the right BUY/HOLD/SELL call, not whether its
generated JSON is byte-identical to the synthetic training target.

In [ ]:
import re

def generate_opinion(system, user, max_new_tokens=80):
    messages = [{"role":"system","content":system}, {"role":"user","content":user}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r'"opinion"\s*:\s*"(BUY|HOLD|SELL)"', text)
    return m.group(1) if m else None, text

correct, total, confused = 0, 0, []
eval_sample = val_examples[:150]  # keep eval wall-clock reasonable on a T4
for ex in eval_sample:
    true_label = json.loads(ex["target"])["opinion"]
    pred, raw_text = generate_opinion(ex["system"], ex["user"])
    total += 1
    if pred == true_label:
        correct += 1
    else:
        confused.append((ex["user"], true_label, pred, raw_text[:120]))

print(f"held-out accuracy on {total} examples: {correct/total:.1%}")
print("\nsample misses:")
for u, t, p, raw in confused[:8]:
    print(f"  true={t} pred={p}  {u[:80]!r}")

## 5. Sanity-check on real headlines from the project's own database

Not a formal eval (no ground truth here — these are just spot checks
against a handful of real AAPL headlines from `historical_headlines`,
copied in manually since this notebook has no access to the user's local
Postgres). Compare these calls against what GPT-4o-mini said for the
same headlines (visible in the `agent_opinions` table / the backtest
dashboard) as a rough transfer check, not a scored metric.

In [ ]:
REAL_HEADLINE_CHECKS = [
    "Apple Fiscal Q3 Earnings Beat Estimates On Strong iPhone, Services Growth",
    "Apple Faces Antitrust Probe Over App Store Practices In EU",
    "Apple Announces New MacBook Pro Lineup At Fall Event",
    "Apple Supplier Warns Of Weaker-Than-Expected iPhone Demand",
    "Apple Stock Ticks Higher In Quiet Trading Ahead Of Fed Decision",
]

for h in REAL_HEADLINE_CHECKS:
    user = f"Symbol: AAPL\nHeadlines (1):\n- {h}"
    pred, raw_text = generate_opinion(SYSTEM_PROMPT, user)
    print(f"{pred:5s} | {h}")
    print(f"       -> {raw_text.strip()[:160]}")
    print()

## Next: bring the adapter back into the project

1. Download `qwen-sentiment-lora-adapter.zip` from this notebook's Output panel.
2. Unzip it into the project repo, e.g. `models/qwen-sentiment-lora/`.
3. Back in `paper-trading-agent`, a new local-inference sentiment backend
   (loading Qwen2.5-1.5B-Instruct + this adapter via `transformers`/`peft`,
   wrapped to match the same call shape `sentiment_analyst.py` already
   uses) replaces the `ChatOpenAI` call for `sentiment_analyst`, gated
   behind a flag so both backends stay swappable.
4. Re-run a backtest with the local model in place of GPT-4o-mini and
   compare against backtests 3, 4, 6 already on the results dashboard —
   does the fine-tuned local model change trading behavior, and does the
   held-out accuracy above translate into a real difference on this
   project's own strategy, not just PhraseBank's.
